In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("ReSplit_Star_Schema_Correct") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 读取清洗后的数据
cleaned_path = "/home/lst/my-spark/cleaned/"     # ← 改成你 cleaned 的实际路径
df = spark.read.parquet(cleaned_path)

print(f"读取 cleaned 数据: {df.count():,} 行")

base_path = "/home/lst/my-spark/star_schema/"    # ← 最后一定要有 /

# 事实表
fact_order_items = df.select(
    "item_id", "increment_id", "created_at", "year", "month",
    "price", "qty_ordered", "total_value", "grand_total", 
    "discount_amount", "status", "payment_method",
    "sku", "Customer ID", "category_name_1"
).dropDuplicates(["item_id"])

# 维度表
dim_products = df.select("sku", "category_name_1").dropDuplicates(["sku"])
dim_customers = df.select("Customer ID", "Customer Since").dropDuplicates(["Customer ID"])
dim_time = df.select("created_at", "year", "month").dropDuplicates(["created_at"])
dim_status = df.select("status").dropDuplicates(["status"])

print("\n拆分后行数：")
print(f"fact_order_items : {fact_order_items.count():,}")
print(f"dim_products     : {dim_products.count():,}")
print(f"dim_customers    : {dim_customers.count():,}")
print(f"dim_time         : {dim_time.count():,}")
print(f"dim_status       : {dim_status.count():,}")

# 保存（确保路径正确）
fact_order_items.write.mode("overwrite").parquet(base_path + "fact_order_items")
dim_products.write.mode("overwrite").parquet(base_path + "dim_products")
dim_customers.write.mode("overwrite").parquet(base_path + "dim_customers")
dim_time.write.mode("overwrite").parquet(base_path + "dim_time")
dim_status.write.mode("overwrite").parquet(base_path + "dim_status")

print("\n✅ 星型模型重新拆分完成！路径正确")
print("保存位置:", base_path)

26/04/17 23:22:10 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/17 23:22:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 23:22:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 23:22:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


读取 cleaned 数据: 584,513 行

拆分后行数：


fact_order_items : 584,513
dim_products     : 84,885
dim_customers    : 115,326
dim_time         : 789
dim_status       : 17



✅ 星型模型重新拆分完成！路径正确
保存位置: /home/lst/my-spark/star_schema/


In [4]:
spark.stop()